# MPNet fixed-schedule follow-up

## Context & Methods
Compare six fixed epochs with the prior at-most-two-epoch schedule across seeds 17, 42 and 73. Keep the original 256 training and 192 validation pages, raw HTML, complete overlapping windows, max-page BCE loss, learning rates and validation checkpoint/threshold rule. Both frozen and fine-tuned encoders remain controls.

### Key Assumptions
The prior seed-42 checkpoint showed weak training and nearly tied window scores. A longer schedule is a hypothesis, not a confirmed fix. Dataset labels are machine generated, and website/template identifiers are unavailable. The new holdout comprises 453 eligible official-test rows from 2100:2600 (217 attacks, 236 non-injections); all older test slices are consumed. No final-test score chooses a model, seed or threshold.

Prior and new fine-tuned packages must reproduce all saved validation decisions on the A100. The first two epochs must reproduce the prior trials before the new holdout can be scored.

### 1. Install the same pinned environment

In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', WANDB_MODE='disabled', TOKENIZERS_PARALLELISM='false')
packages = ['torch==2.9.1','sentence-transformers==5.1.2','transformers==4.57.3','datasets==4.4.1','xgboost==3.1.2','scikit-learn==1.7.2','numpy==2.3.5','scipy==1.16.3','beautifulsoup4==4.14.2','accelerate==1.11.0']
for command in [[sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'], [sys.executable,'-m','pip','install','-q'] + packages]:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Environment setup failed')
print('Pinned environment installed. Model training uses fresh Python subprocesses.')


### 2. Verify the requested A100

In [ ]:
import subprocess, sys, json
code = "import torch,json; assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(); print(json.dumps({'gpu':torch.cuda.get_device_name(),'memory_GiB':torch.cuda.get_device_properties(0).total_memory/2**30}))"
print(subprocess.check_output([sys.executable,'-c',code],text=True))


## Data
### 3. Verify frozen inputs and prior checkpoints

In [ ]:
import tarfile, hashlib, json, time
from pathlib import Path
archive=Path('/content/supervised_schedule_inputs.tar.gz')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='2c4ca910b17175bb70dff40719ac2ea7d49c900550918eb0682e678413c606bc'
output=Path('/content/mpnet_supervised_schedule')
if not output.exists():
    with tarfile.open(archive) as bundle:
        bundle.extractall('/content',filter='data')
deadline=time.monotonic()+1800
while not Path('/content/prior_weights_ready.json').exists():
    if time.monotonic()>deadline: raise TimeoutError('Prior checkpoint upload did not finish')
    print('Waiting for the three prior checkpoint uploads...',flush=True)
    time.sleep(20)
for seed in [17,42,73]:
    source=Path(f'/content/prior_finetuned_{seed}.pt')
    target=output/'prior'/f'finetuned_{seed}'/'checkpoint.pt'
    if source.exists(): source.replace(target)
protocol=json.loads((output/'protocol.json').read_text())
for name, expected in protocol['files'].items():
    with (output/name).open('rb') as stream:
        assert hashlib.file_digest(stream,'sha256').hexdigest()==expected,name
print('Frozen counts:',protocol['counts'])
print('Frozen windows:',protocol['windows'])
print('Frozen at:',protocol['frozen_at'])


### 4. Reproduce the prior packages on validation

In [ ]:
import subprocess, sys, os

def run_stage(stage, command):
    with Path(f'/content/schedule_{stage}.log').open('w') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=dict(os.environ,USE_TF='0',USE_FLAX='0',WANDB_MODE='disabled')) as process:
            for line in process.stdout:
                log.write(line); log.flush(); print(line,end='',flush=True)
            if process.wait(): raise RuntimeError(stage+' failed')

for seed in [17,42,73]:
    folder=output/'prior'/f'finetuned_{seed}'
    run_stage(f'prior_replay_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'replay','--package',str(folder),'--input',str(output/'validation.jsonl'),'--expected',str(folder/'validation_predictions.jsonl'),'--output',str(folder/'package_replay.json')])


### 5. Train all six trials for six fixed epochs
Only training and validation influence checkpoint selection. Diagnostics record gradients, optimizer steps, and selected training windows.

In [ ]:
for seed in [17,42,73]:
    report=json.loads((output/'prior'/f'finetuned_{seed}'/'package_replay.json').read_text())
    assert report['n']==192 and report['identical_decisions'] and report['max_absolute_score_error']<=2e-6
run_stage('train',[sys.executable,'-u',str(output/'supervised_pilot.py'),'train','--output',str(output)])


### 6. Verify prefix reproduction and new inference packages

In [ ]:
assert (output/'selection_frozen.json').exists()
run_stage('prefix',[sys.executable,'-u',str(output/'supervised_schedule.py'),'prefix-check','--output',str(output)])
for seed in [17,42,73]:
    folder=output/f'finetuned_{seed}'
    run_stage(f'export_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'export','--package',str(folder),'--assets',str(output/'prior/finetuned_17/inference_assets')])
    run_stage(f'new_replay_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'replay','--package',str(folder),'--input',str(output/'validation.jsonl'),'--expected',str(folder/'validation_predictions.jsonl'),'--output',str(folder/'package_replay.json')])


### 7. Evaluate the fresh holdout
All twelve old/new checkpoints and thresholds must be frozen. The comparison includes every seed.

In [ ]:
run_stage('evaluate',[sys.executable,'-u',str(output/'supervised_schedule.py'),'evaluate','--output',str(output)])


## Results
### 8. Compare schedules without selecting a winning seed

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
result=json.loads((output/'holdout_results.json').read_text())
assert len(result['summaries'])==len(result['prior_summaries'])==6
for seed in [17,42,73]:
    for label,key in [('Prior schedule','prior_summaries'),('Six epochs','summaries')]:
        row=result[key][f'finetuned_{seed}']
        print(f"{seed} {label}: AUC={row['roc_auc']:.4f}, recall={row['recall']['value']:.2%}, FPR={row['fpr']['value']:.2%}")
fig,axes=plt.subplots(1,3,figsize=(12,3.7),layout='constrained')
for ax,metric,title in zip(axes,['roc_auc','recall','fpr'],['ROC-AUC','Attack recall','False-positive rate']):
    for offset,(label,key,color,marker) in zip([-.12,.12],[('Prior schedule','prior_summaries','#2364aa','o'),('Six epochs','summaries','#b24c16','s')]):
        values=[result[key][f'finetuned_{seed}'][metric] for seed in [17,42,73]]
        values=[v['value'] if isinstance(v,dict) else v for v in values]
        ax.scatter([i+offset for i in range(3)],values,label=label,color=color,marker=marker)
    ax.set(title=title,xticks=range(3),xticklabels=['17','42','73'],xlabel='Training seed',ylim=(0,1))
    if metric!='roc_auc': ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y',alpha=.2)
axes[0].legend()
axes[2].axhline(.05,color='#555555',ls='--',lw=1,label='Validation budget (5%)')
axes[2].legend(fontsize=8)
fig.suptitle('Fresh BrowseSafe holdout: 453 pages; all validation-frozen operating points')
fig.savefig(output/'schedule_comparison.png',dpi=180)
plt.show()
print('Nominal paired intervals and independent replay are computed after artifact retrieval.')


## Takeaways
Report recall together with actual false positives for every seed. A validation constraint is not a population guarantee. This is a BrowseSafe-only research experiment; unknown template relationships and source-label uncertainty remain. The 453-page holdout is consumed once evaluated.

Download all manifest files and logs before stopping the A100 session. Prior checkpoint files can be restored from their verified existing local copies. Avoid duplicating large result archives on the Mac.